In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM
from transformers import XLMRobertaTokenizer, TFAutoModel
import matplotlib.pyplot as plt
#from keras.utils import plot_model
import os
from datasets import load_dataset
import torch
from huggingface_hub import from_pretrained_keras
from collections import Counter
import tensorflow as tf
from tensorflow.keras import optimizers
from tensorflow.keras import metrics
from tensorflow.keras import losses

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
#load the data
df_train = pd.read_pickle("Data/df_preprocessed_train_text.pickle").reset_index(drop = True)
df_test = pd.read_pickle("Data/df_preprocessed_test_text.pickle").reset_index(drop = True)

In [2]:
#1. design a model which calls xlm-roberta to take in preprocessed text and produce vectors, freeze this part of the model to prevent training
#2. take in the vectorized output of step (1) in the lstm layer
#3. propegate the outputs of each layer through the model, producing a meaningfull representation of the case
#4. combine with the model for numeric and categorical variables
#5. make a final regression

#specify metrics (this can me multiple, see https://keras.io/api/metrics/)
#specify optimizer (note: there are many optimizers with many arguments which might be worthwhile looking into) --> https://keras.io/api/optimizers/
#specify loss metric (note: many losss functions, see https://keras.io/api/losses/)

In [4]:
def extract_text(df):
    text_columns = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]
    text_per_col_dict= {}
    for col in text_columns:
        list_of_text = df[col].to_list()
        text_per_col_dict[col] = list_of_text
    return text_per_col_dict

In [5]:
def extract_target_var(df, target_col = "award_contract/val_total: 0"):
    y = df[target_col].astype(int).values
    return y

In [6]:
def tokenize(text_per_col_dict_train, size):
    language_model_name = "jplu/tf-xlm-roberta-base"
    tokenizer = XLMRobertaTokenizer.from_pretrained(language_model_name)

    text_per_col_tokenized = {}
    for col in list(text_per_col_dict_train.keys())[:1]:
        tokenized_data = tokenizer(text_per_col_dict_train[col][:size], 
                                          return_tensors="tf",
                                          max_length=128,
                                          padding="max_length",
                                          truncation=True)
        text_per_col_tokenized[col] = tokenized_data
    return text_per_col_tokenized

In [7]:
def map_to_labels(data):
    label_list = []
    encoding_list = []
    for value in data:
        if value <= 50000:
            label_list.append("<50k")
            encoding_list.append(0)
        if value > 50000 and value <= 100000:
            label_list.append(">50k<100k")
            encoding_list.append(1)
        elif value > 100000 and value <= 200000:
            label_list.append(">100k<200k")
            encoding_list.append(2)
        elif value > 200000 and value <= 300000:
            label_list.append(">200k<300K")
            encoding_list.append(3)
        elif value > 300000 and value <= 400000:
            label_list.append(">300K<400k")
            encoding_list.append(4)
        elif value > 400000 and value <= 800000:
            label_list.append(">400k<800k")
            encoding_list.append(5)
        elif value > 800000 and value <= 1000000:
            label_list.append(">800k<1m")
            encoding_list.append(6)
        elif value > 1000000 and value <= 2000000:
            label_list.append(">1m<2m")
            encoding_list.append(7)
        elif value > 2000000 and value <= 3000000:
            label_list.append(">2m<3m")
            encoding_list.append(8)
        elif value >= 3000000: 
            label_list.append(">3m")
            encoding_list.append(9)

    label_array = np.array(label_list)
    encoding_array = np.array(encoding_list)
    return label_array, encoding_array

In [8]:
#get a dict containing all the text from the training set
text_per_col_dict_train = extract_text(df_train)
text_per_col_dict_test = extract_text(df_test)
tokenized_text_train = tokenize(text_per_col_dict_train, size = 1000)
tokenized_text_test = tokenize(text_per_col_dict_test, size = 200)
y_train = extract_target_var(df_train[:1000])
y_test = extract_target_var(df_test[:200])

In [ ]:
y_train_labels, y_train_encoding = map_to_labels(y_train)
y_test_labels, y_test_encoding = map_to_labels(y_test)
Counter(y_train_labels)

In [ ]:
from transformers import TFAutoModelForSequenceClassification
language_model_name = "jplu/tf-xlm-roberta-base"
language_model = TFAutoModelForSequenceClassification.from_pretrained(language_model_name, num_labels=10)

In [13]:
#specify model parameters
learning_rate = 0.01
optimizer = "Adam"
loss = losses.SparseCategoricalCrossentropy(from_logits=True)

language_model.compile(optimizer=optimizer,
                       loss=loss,
                       metrics=metrics.SparseCategoricalAccuracy())

In [14]:
input_ids = tokenized_text_train["short_descr"]["input_ids"]
attention_mask = tokenized_text_train["short_descr"]["attention_mask"]

In [ ]:
language_model.fit(x = [input_ids, attention_mask], y=y_train_encoding,
                   validation_data=([tokenized_text_test["short_descr"]], y_test_encoding),
    epochs=10, batch_size = 30)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("tweet_eval", "emotion")

In [ ]:
dataset

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)
# Tokenize entire dataset 
tokenized_dataset = dataset.map(tokenize, batched=True, batch_size=None)

In [ ]:
tokenized_dataset

In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

batch_size=64

tf_train_dataset = tokenized_dataset["train"].to_tf_dataset(
    columns=["text", "labels", "inputs_ids", "attention_mask"], 
    label_cols=["label"], 
    shuffle=True, 
    batch_size=batch_size,
    collate_fn=data_collator
)

In [ ]:
#initialize model constants
XML_roberta_vocabulary = 30522 #look up in huggingface documentation
max_features = XML_roberta_vocabulary
embedding_dim = 258 #max length of a token vector is 258

#specify the model and use the tokenizer output as input for the language model. complement language model with regression head
language_model = TFAutoModel.from_pretrained(language_model_name)
language_model.trainable = False

embeddings = language_model(**tokenized_data, output_hidden_states = True)

input_embeddings = Input(shape = (128,), name="language embeddings")
embeddings = language_model(**tokenized_data, output_hidden_states = True)(input_embeddings)
regression_layer = Dense(1, activation="linear")(input_embeddings)
regression_model = Model(inputs=input_embeddings, outputs=regression_layer)

#specify model parameters
learning_rate = 0.01
optimizer = "Adam"
loss = "mean_absolute_error"
metrics = ["mean_absolute_error", "mean_squared_error"]

regression_model.compile(optimizer=optimizer,
                         loss=loss,
                         metrics=metrics)

In [ ]:
# Tokenization (done separately)
train_tokenized_data = tokenizer(train_texts, return_tensors="tf", ...)
train_labels = # Your actual labels for the training set

In [118]:
#import embedded data
with open("Data/word vectors/X_test_embedded_10000_20000", "rb") as f:
    embeddings_test = pickle.load(f)

with open("Data/word vectors/X_train_embedded_10000_20000", "rb") as f:
    embeddings_train = pickle.load(f)

In [128]:
embeddings_test = embeddings_test[:2000]

In [136]:
tf.debugging.set_log_device_placement(True)
input_embeddings = Input(shape=(128, 252), name="embedding input")
flatten = Flatten()(input_embeddings)
dense_layer = Dense(128, activation="relu")(flatten)
dense_layer = Dense(32, activation = "relu")(dense_layer)
classification_layer = 
language_model = Model(inputs=input_embeddings,
                         outputs=final_layer)


input_

# Compile the model
learning_rate = 0.1
optimizer = "Adam"
loss = "mean_absolute_error"
metrics = ["mean_absolute_error", "mean_squared_error"]

regression_model.compile(loss=loss, optimizer=optimizer)

In [ ]:
regression_model.fit(
    x=[embeddings_train], y=y_train,
    validation_data=([embeddings_test], y_test),
    epochs=10, batch_size = 200)

In [ ]:
#now let's make a model for our categorical and numerical data
#define variables
optimizer = keras.optimizers.Adam(learning_rate=0.01) #tf.keras.optimizers.experimental.Adagrad(learning_rate=0.001)
loss_function = "mean_absolute_error"

#define the numeric and categorical model
input_dim_num_cat = 55 #len(X_train[0])
input_num_cat = Input(shape=input_dim_num_cat)
num_cat_layer_1 = Dense(128, activation = "relu")(input_num_cat)
num_cat_layer_2 = Dense(64, activation="relu")(num_cat_layer_1)
num_cat_layer_3 = Dense(32, activation="relu")(num_cat_layer_2)
num_cat_layer_4 = Dense(10, activation="relu")(num_cat_layer_3)
#num_cat_layer_5 = Dense(4, activation="relu")(num_cat_layer_4)
#num_cat_layer_6 = Dense(1, activation="linear")(num_cat_layer_5)
#model_num_cat = Model(input_num_cat, num_cat_layer_6)
#model_num_cat.compile(loss=loss_function, optimizer=optimizer)
#model_num_cat.fit(x=[X_train], y=y_train,
#                  validation_data=([X_test], y_test),
#                  epochs=100, batch_size = 68)